# Discover and download CPG0016 profiles CSVs
This notebook shows how to build a manifest of the CPG0016 analysis/profile CSV paths from S3, reuse a cached manifest locally, and download one or all profile CSV groups while preserving their source-relative directory structure.

In [ ]:
from pathlib import Path

from jump_image_datasets.cpg0016 import CPG0016AnalysisCSVDownloader

## Build the profiles manifest
The downloader discovers the available analysis/profile CSV paths from S3 and saves a manifest CSV locally for reuse.

In [ ]:
downloader = CPG0016AnalysisCSVDownloader(
    manifest_download_dir=Path("profiles_manifest_cache"),
    manifest_csv_path=Path("downloaded_cpg0016_profiles_paths.csv"),
    parallel=True,
    workers=8,
)
profiles_df = downloader.get_dataframe()
print(f"Profiles manifest shape: {profiles_df.shape}")
profiles_df.head()

## Reuse a cached manifest
Skip the S3 discovery pass when you already have the manifest files locally.

In [ ]:
cached_downloader = CPG0016AnalysisCSVDownloader(
    manifest_download_dir=Path("profiles_manifest_cache"),
    manifest_csv_path=Path("downloaded_cpg0016_profiles_paths.csv"),
    use_existing_manifest_without_s3_check=True,
    parallel=True,
    workers=8,
)
cached_profiles_df = cached_downloader.get_dataframe().iloc[:10].copy()

## Download one profile CSV group
Download one manifest column while preserving the S3-relative directory tree under a per-row output root to avoid filename collisions.

In [ ]:
cached_profiles_df["OutputRoot"] = "downloaded_profiles_csvs"
nuclei_summary = cached_downloader.download_csvs_from_column(
    dataframe=cached_profiles_df,
    column_name="Nuclei_S3_Path",
    output_root_column="OutputRoot",
    parallel=True,
    workers=8,
)
print(nuclei_summary)

## Download all profile CSV groups
Download all three analysis/profile CSV types from the same filtered manifest DataFrame.

In [ ]:
all_profiles_summary = cached_downloader.download_csvs_from_columns(
    dataframe=cached_profiles_df,
    output_root_column="OutputRoot",
    parallel=True,
    workers=8,
)
print(all_profiles_summary)